# UCI 603 In-Vehicle Coupon — Causal BN (B0 / H1 / H2)

Môn IT701100, HK1 2026–2027. Kế hoạch: `../research-plan-coupon.md`. Hồ sơ dataset: `../docs/dataset-scouting/05_uci603_coupon_profile.md`.

**Câu hỏi.** Với một người lái xe trong một bối cảnh cụ thể được hệ gợi ý trong xe đề xuất coupon giảm 20%, nếu coupon hết hạn sau **2 giờ** (T=1) thay vì **1 ngày** (T=0) thì xác suất chấp nhận coupon (Y) thay đổi bao nhiêu?

**Estimand.** ATE = P(Y=1 | do(T=1)) − P(Y=1 | do(T=0)) trên TEST reference cohort; thêm CATE theo `coupon`.

Cấu trúc notebook theo slide 76: 1 câu hỏi và giả định · 2 nguồn dữ liệu và audit · 3 tiền xử lý và split · 4 B0 · 5 H1 · 6 H2 · 7 đánh giá · 8 kết luận. **Notebook này hiện hoàn thành mục 1–3 (Phase 1–2).**

## 1. Setup

In [1]:
import sys, json, time, warnings, logging
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import prep_coupon as pc
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
import platform, pgmpy, sklearn, torch
ENV = {"python": sys.version.split()[0], "platform": platform.platform(), "numpy": np.__version__,
       "pandas": pd.__version__, "pgmpy": pgmpy.__version__, "sklearn": sklearn.__version__, "torch": torch.__version__}
try:
    import dowhy; ENV["dowhy"] = dowhy.__version__
except ImportError:
    ENV["dowhy"] = "chưa cài"
pd.Series(ENV).to_frame("value")

,value
python,3.14.6
platform,macOS-27.0.1-arm64-arm-64bit-Mach-O
numpy,2.5.4
pandas,3.1.0
pgmpy,1.1.2
sklearn,1.9.1
torch,2.15.0
dowhy,0.8


## 2. Nguồn dữ liệu, timing và bảng giả định (slide 54)

| Mục | Giá trị |
|---|---|
| Nguồn | UCI ID 603, DOI 10.24432/C5GS4P, license CC BY 4.0 |
| Bài gốc | Wang, Rudin, Doshi-Velez, Liu, Klampfl, MacNeille (2017), JMLR 18(70) |
| File | `data/raw/coupon/in-vehicle-coupon-recommendation.csv` (checksum in ở ô dưới) |
| Đơn vị quan sát | một cặp (người trả lời, kịch bản lái xe); mỗi người ~20–22 kịch bản |
| T | `expiration`: 1d = 0, 2h = 1; đặt bởi phiên bản bảng hỏi lúc hiển thị coupon |
| Y | chấp nhận coupon ("đi ngay" hoặc "đi sau, trước khi hết hạn") |

**Giả định (A1–A4).**
- A1. T được gán theo phiên bản bảng hỏi: độc lập với người trả lời, phụ thuộc biến kịch bản. Kiểm tra ở mục 3.4.
- A2. Sau khi điều chỉnh biến kịch bản, không còn confounder ẩn giữa T và Y (phiên bản bảng hỏi không có trong dữ liệu).
- A3. Y là ý định khai báo trong tình huống giả định; kết luận chỉ nói về hành vi khai báo.
- A4. Các dòng cùng một người không độc lập; xử lý bằng split theo người (`resp_id`), không bằng mô hình.

In [2]:
raw = pc.load_raw()
src = {"file": str(pc.RAW.relative_to(ROOT)), "sha256": pc.sha256(pc.RAW), "rows": len(raw), "cols": raw.shape[1] - 1,
       "downloaded": "2026-11-06", "license": "CC BY 4.0", "doi": "10.24432/C5GS4P"}
pd.Series(src).to_frame("value")

,value
file,data/raw/coupon/in-vehicle-coupon-recommendati...
sha256,d47897367f5a12a31dece0f4e42d7bd9da76d7016b568d...
rows,12684
cols,26
downloaded,2026-11-06
license,CC BY 4.0
doi,10.24432/C5GS4P


### 2.1 Audit file gốc: mức, thiếu, hằng số, trùng

In [3]:
audit_raw = pc.audit_raw(raw)
print("missing:", audit_raw["missing_counts"])
print("constant:", audit_raw["constant_cols"], "| exact duplicates:", audit_raw["exact_duplicate_rows"])
print(f"Y rate = {audit_raw['y_rate']:.3f} | P(expiration=2h) = {audit_raw['t_rate_2h']:.3f}")
pd.Series(audit_raw["levels"], name="n_levels").to_frame().T

missing: {'car': 12576, 'Bar': 107, 'CoffeeHouse': 217, 'CarryAway': 151, 'RestaurantLessThan20': 130, 'Restaurant20To50': 189}
constant: ['toCoupon_GEQ5min'] | exact duplicates: 74
Y rate = 0.568 | P(expiration=2h) = 0.441


,destination,passenger,weather,temperature,time,coupon,expiration,gender,age,maritalStatus,has_children,education,occupation,income,car,Bar,CoffeeHouse,CarryAway,RestaurantLessThan20,Restaurant20To50,toCoupon_GEQ5min,toCoupon_GEQ15min,toCoupon_GEQ25min,direction_same,direction_opp,Y
n_levels,3,4,3,3,5,5,2,2,8,5,2,6,25,9,6,6,6,6,6,6,1,2,2,2,2,2


### 2.2 Khác biệt giữa file và phụ lục B của bài báo
- Phụ lục ghi `time` có 3 mức; file có 5 (thêm 7AM, 10PM). Dùng file làm chuẩn.
- Cột `passanger` sai chính tả trong file gốc, đổi thành `passenger` khi nạp.
- `car` thiếu 99%, `toCoupon_GEQ5min` là hằng số, `direction_opp` = 1 − `direction_same`: bỏ.

## 3. Tiền xử lý và split (slide 55)

### 3.1 Tái tạo `resp_id`
Không có cột ID người trả lời. Gom các dòng **liên tiếp** có cùng 13 cột nhân khẩu học và thói quen thành một block. Bài báo nói mỗi người trả lời 20 kịch bản; dữ liệu cho thấy đa số block có 22 dòng.

In [4]:
d_all = pc.recode(raw)
blocks = d_all.groupby("resp_id").size()
print(f"blocks = {blocks.shape[0]}; distinct respondent keys = {raw[pc.RESP_COLS].drop_duplicates().shape[0]}")
blocks.value_counts().sort_index().rename("n_blocks").to_frame().T

blocks = 587; distinct respondent keys = 567


,6,14,16,17,19,20,21,22,23,28,29,34
n_blocks,12,1,2,7,1,8,29,517,4,1,1,4


### 3.2 Eligibility flow
Loại ô Bar × Work (0% hạn 2h, không có overlap) và dòng thiếu tần suất đến đúng loại quán của coupon. Dòng trùng hoàn toàn được giữ (có thể là hai người cùng hồ sơ) và chỉ ghi nhận.

In [5]:
d, flow = pc.eligibility(raw, d_all)
pd.DataFrame(flow)

,step,n,removed,duplicates_present
0,raw UCI file,12684,NaN,NaN
1,"loại ô Bar × Work (0% T=2h, không có overlap)",12197,487.0,NaN
2,loại dòng thiếu tần suất đến đúng loại quán củ...,12037,160.0,NaN
3,"dòng trùng hoàn toàn (giữ, chỉ ghi nhận)",12037,0.0,73.0


In [6]:
print(f"Sau eligibility: n = {len(d)}, người = {d['resp_id'].nunique()}, Y rate = {d['Y'].mean():.4f}, P(T=1) = {d['T'].mean():.4f}")

Sau eligibility: n = 12037, người = 580, Y rate = 0.5767, P(T=1) = 0.4583


### 3.3 Từ điển biến: mức sau mã hoá, timing, vai trò trong DAG
Nút DAG (12): tầng 1 người trả lời → tầng 2 kịch bản → T → Y. `habit_match` = tần suất đến đúng loại quán của coupon đang hiển thị (gộp 5 cột thói quen).

In [7]:
vd = pc.variable_dictionary(d)
vd.style.set_properties(**{"text-align": "left"}) if hasattr(vd, "style") else vd

,variable,n_levels,levels,timing,role
0,habit_match,3,"low=4925, mid=3750, high=3362","người trả lời, trước T",baseline C (→Y); cấm →T
1,age_bin,4,"26-35=4342, <26=3033, 36-50=2956, 50+=1706","người trả lời, trước T",baseline C (→Y); cấm →T
2,income_bin,3,"<37.5k=4621, 37.5-75k=4116, 75k+=3300","người trả lời, trước T",baseline C (→Y); cấm →T
3,gender,2,"Female=6193, Male=5844","người trả lời, trước T",baseline C (→Y); cấm →T
4,has_children,2,"0=7073, 1=4964","người trả lời, trước T",baseline C (→Y); cấm →T
5,coupon,5,"Coffee House=3924, Restaurant(<20)=2759, Carry out & Take away=2371, Bar=1513, Restaurant(20-50)=1470","thiết kế kịch bản, trước T","confounder C_core (→T, →Y; positivity đầy đủ)"
6,destination,3,"No Urgent Place=6184, Home=3197, Work=2656","thiết kế kịch bản, trước T","confounder C_core (→T, →Y; positivity đầy đủ)"
7,time_bin,3,"evening=5166, morning=4894, afternoon=1977","thiết kế kịch bản, trước T","confounder C_ext (→T, →Y; có ô trống, chỉ độ nhạy)"
8,passenger,4,"Alone=6748, Friend(s)=3249, Partner=1054, Kid(s)=986","thiết kế kịch bản, trước T",C phụ (→Y)
9,weather,3,"Sunny=9635, Snowy=1305, Rainy=1097","thiết kế kịch bản, trước T","confounder C_ext (→T, →Y; có ô trống, chỉ độ nhạy)"


In [8]:
print("tiers:", json.dumps(pc.TIERS, ensure_ascii=False))
print("C_core:", pc.C_CORE, "| C_ext:", pc.C_EXT)
print("C_full:", pc.C_FULL)

tiers: {"respondent": ["habit_match", "age_bin", "income_bin", "gender", "has_children"], "scenario": ["coupon", "destination", "time_bin", "passenger", "weather", "dist", "direction_same"], "treatment": ["T"], "outcome": ["Y"]}
C_core: ['coupon', 'destination'] | C_ext: ['coupon', 'destination', 'time_bin', 'weather']
C_full: ['coupon', 'destination', 'time_bin', 'weather', 'passenger', 'dist', 'direction_same', 'habit_match', 'age_bin', 'income_bin', 'gender', 'has_children']


### 3.4 Kiểm tra giả định A1: T độc lập với người trả lời, phụ thuộc kịch bản
Biên độ của P(T=1) giữa các mức của từng biến. Biến người trả lời phải có biên độ gần 0; biến kịch bản có thể lớn.

In [9]:
ti = pd.DataFrame(pc.t_independence(d)).T
ti["tier"] = ["respondent" if i in pc.TIERS["respondent"] else "scenario" for i in ti.index]
ti.sort_values(["tier", "range"], ascending=[False, False])

,min_pT1,max_pT1,range,tier
coupon,0.302,0.556,0.254,scenario
weather,0.303,0.549,0.246,scenario
time_bin,0.354,0.496,0.142,scenario
dist,0.418,0.525,0.107,scenario
passenger,0.422,0.527,0.105,scenario
direction_same,0.450,0.489,0.040,scenario
destination,0.435,0.470,0.035,scenario
habit_match,0.442,0.478,0.036,respondent
age_bin,0.447,0.475,0.027,respondent
income_bin,0.443,0.469,0.026,respondent


### 3.5 Split theo `resp_id`, 5 seed, 70/15/15
Stratify theo tertile tỉ lệ Y trung bình của mỗi người. Chỉ số `row_id` lưu ở `artifacts/coupon/splits.json` và dùng chung cho B0/H1/H2.

In [10]:
splits = pc.make_splits(d)
pd.DataFrame({s: {k: len(v) for k, v in sp.items() if k != "n_resp"} | {f"resp_{k}": n for k, n in sp["n_resp"].items()}
              for s, sp in splits.items()}).T

,train,val,test,resp_train,resp_val,resp_test
0,8394,1819,1824,405,87,88
1,8440,1784,1813,405,87,88
2,8372,1834,1831,405,87,88
3,8399,1842,1796,405,87,88
4,8368,1830,1839,405,87,88


### 3.6 Overlap (positivity) trên TRAIN
`C_core` = (coupon, destination): 14 ô, mọi ô có ≥ 89 dòng mỗi nhánh T trên cả 5 seed. `C_ext` thêm `time_bin` và `weather` (cả hai →T mạnh) nhưng tạo 27 ô trống (Bar × No Urgent × afternoon, Bar × Snowy, Carry out × Snowy, Restaurant(<20) × Rainy...). Vì vậy identify dùng `C_core`; `C_ext` chỉ dùng cho phân tích độ nhạy và cho dự đoán.

In [11]:
tr0 = d[d["row_id"].isin(splits["0"]["train"])]
ov_core = pc.overlap_table(tr0, pc.C_CORE)
ov_core.round(3)

,coupon,destination,n_T0,n_T1,y_rate_T0,y_rate_T1,min_arm,p_T1
0,Bar,Home,243,102,0.506,0.363,102,0.296
1,Bar,No Urgent Place,490,210,0.398,0.424,210,0.300
2,Carry out & Take away,Home,312,152,0.804,0.776,152,0.328
3,Carry out & Take away,No Urgent Place,353,297,0.827,0.704,297,0.457
4,Carry out & Take away,Work,354,186,0.718,0.554,186,0.344
5,Coffee House,Home,301,342,0.445,0.269,301,0.532
6,Coffee House,No Urgent Place,686,773,0.638,0.534,686,0.530
7,Coffee House,Work,236,410,0.572,0.376,236,0.635
8,Restaurant(20-50),Home,322,133,0.491,0.226,133,0.292
9,Restaurant(20-50),No Urgent Place,142,139,0.585,0.388,139,0.495


In [12]:
ov_ext = pc.overlap_table(tr0, pc.C_EXT)
summary = {}
for s in pc.SEEDS:
    tr = d[d["row_id"].isin(splits[str(s)]["train"])]
    for name, cset in [("C_core", pc.C_CORE), ("C_ext", pc.C_EXT)]:
        ov = pc.overlap_table(tr, cset)
        summary[(s, name)] = {"cells": len(ov), "min_arm": int(ov["min_arm"].min()),
                              "cells<30": int((ov["min_arm"] < 30).sum()), "empty_arm": int((ov["min_arm"] == 0).sum())}
pd.DataFrame(summary).T

cells  min_arm  cells<30  empty_arm
0 C_core     14      100         0          0
  C_ext      54        0        34         27
1 C_core     14       89         0          0
  C_ext      54        0        32         27
2 C_core     14      102         0          0
  C_ext      54        0        35         27
3 C_core     14       95         0          0
  C_ext      54        0        33         27
4 C_core     14       91         0          0
  C_ext      54        0        34         27

In [13]:
print("Ô trống của C_ext trên TRAIN seed 0 (cần loại khỏi reference cohort nếu điều chỉnh C_ext):")
ov_ext.loc[ov_ext["min_arm"] == 0, pc.C_EXT + ["n_T0", "n_T1"]].to_string(index=False)

Ô trống của C_ext trên TRAIN seed 0 (cần loại khỏi reference cohort nếu điều chỉnh C_ext):


'               coupon     destination  time_bin weather  n_T0  n_T1\n                  Bar            Home   evening   Rainy  66.0   0.0\n                  Bar No Urgent Place afternoon   Snowy  67.0   0.0\n                  Bar No Urgent Place afternoon   Sunny  12.0   0.0\n                  Bar No Urgent Place   evening   Snowy  39.0   0.0\n                  Bar No Urgent Place   morning   Rainy  62.0   0.0\nCarry out & Take away            Home   evening   Rainy  22.0   0.0\nCarry out & Take away            Home   evening   Snowy  28.0   0.0\nCarry out & Take away No Urgent Place afternoon   Rainy   0.0  27.0\nCarry out & Take away No Urgent Place afternoon   Sunny 122.0   0.0\nCarry out & Take away No Urgent Place   evening   Snowy  50.0   0.0\nCarry out & Take away No Urgent Place   evening   Sunny  78.0   0.0\nCarry out & Take away No Urgent Place   morning   Rainy   0.0  22.0\nCarry out & Take away No Urgent Place   morning   Snowy  50.0   0.0\nCarry out & Take away            

### 3.7 Mốc sơ bộ trên toàn dữ liệu (chưa dùng TEST, chỉ để định hướng)
Chênh lệch thô và ATE chuẩn hoá theo công thức backdoor thủ công. Khoảng cách nhỏ giữa hai con số là hệ quả của A1 (T gần như được gán ngẫu nhiên trong kịch bản), không phải hạn chế của dữ liệu.

In [14]:
prev = pd.DataFrame({k: pc.naive_and_standardized(d, cset) for k, cset in [("C_core", pc.C_CORE), ("C_ext", pc.C_EXT)]}).T
cate = d.groupby("coupon").apply(lambda x: x.loc[x["T"] == 1, "Y"].mean() - x.loc[x["T"] == 0, "Y"].mean()).rename("naive CATE")
display(prev); cate.round(3).to_frame().T

,naive_diff,standardized_ATE,cells,cells_with_both_T,rows_covered_frac
C_core,-0.15,-0.1639,14.0,14.0,1.000
C_ext,-0.15,-0.1708,54.0,27.0,0.842


coupon,Bar,Carry out & Take away,Coffee House,Restaurant(20-50),Restaurant(<20)
naive CATE,-0.031,-0.12,-0.152,-0.223,-0.243


### 3.8 Lưu artifact Phase 1

In [15]:
raw_, d_, audit, splits_ = pc.main()
print("processed:", pc.PROC / "coupon_discretized.csv", "| rows:", len(d_))
print("artifacts:", sorted(p.name for p in pc.ART.iterdir()))

processed: /Users/tuanla/workspace/master/probabilistic-graphical-models/data/processed/coupon_discretized.csv | rows: 12037
artifacts: ['audit.json', 'config_coupon.json', 'overlap_train_seed0_C_core.csv', 'overlap_train_seed0_C_ext.csv', 'overlap_train_seed1_C_core.csv', 'overlap_train_seed1_C_ext.csv', 'overlap_train_seed2_C_core.csv', 'overlap_train_seed2_C_ext.csv', 'overlap_train_seed3_C_core.csv', 'overlap_train_seed3_C_ext.csv', 'overlap_train_seed4_C_core.csv', 'overlap_train_seed4_C_ext.csv', 'splits.json', 'variable_dictionary.csv']


## 4–7. B0 / H1 / H2 / Đánh giá
Phase 3 trở đi. Việc cần làm trước khi bắt đầu B0:
- DoWhy ≥ 0.12 cần Python < 3.14; `.venv` hiện dùng Python 3.14 nên chỉ cài được DoWhy 0.8. Tạo venv Python 3.13 (hoặc chạy phần DoWhy trên Colab) trước Phase 3.
- Quyết định cách xử lý ô trống của `C_ext` trong do-query: giới hạn reference cohort về các profile có overlap trên TRAIN, hoặc chỉ identify với `C_core`.